# ****#Read data and create DF

In [3]:
customers_raw = spark.read.parquet("abfss://SravyaWS_DEV@onelake.dfs.fabric.microsoft.com/EcommerceLH.Lakehouse/Files/Bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://SravyaWS_DEV@onelake.dfs.fabric.microsoft.com/EcommerceLH.Lakehouse/Files/Bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://SravyaWS_DEV@onelake.dfs.fabric.microsoft.com/EcommerceLH.Lakehouse/Files/Bronze/payments.parquet")
support_raw = spark.read.parquet("abfss://SravyaWS_DEV@onelake.dfs.fabric.microsoft.com/EcommerceLH.Lakehouse/Files/Bronze/support_tickets.parquet")
web_raw = spark.read.parquet("abfss://SravyaWS_DEV@onelake.dfs.fabric.microsoft.com/EcommerceLH.Lakehouse/Files/Bronze/web_activities.parquet")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 5, Finished, Available, Finished, False)

In [3]:
display(customers_raw.limit(5))

StatementMeta(, 48a16b6d-e2da-4323-848f-dce960b8089b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9f9a2d51-a6c4-481d-ada5-81acb0efa5d9)

In [4]:
display(orders_raw.limit(5))

StatementMeta(, 48a16b6d-e2da-4323-848f-dce960b8089b, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72171f9b-0c20-419e-a619-9ae906aa1ff2)

In [5]:
display(payments_raw.limit(5))

StatementMeta(, 48a16b6d-e2da-4323-848f-dce960b8089b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f30808e7-51ff-4fd2-aded-d7e7c614fc46)

In [9]:
display(support_raw.limit(5))

StatementMeta(, 48a16b6d-e2da-4323-848f-dce960b8089b, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3be4d42a-c44f-4aa0-916d-3dfaa119d17d)

In [10]:
display(web_raw.limit(5))

StatementMeta(, 48a16b6d-e2da-4323-848f-dce960b8089b, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9fc3d587-3a3d-4b52-a379-fed89740360a)

Create Delta Bronzetable

In [4]:
# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 6, Finished, Available, Finished, False)

In [14]:
%%sql 
select * from customers limit 5

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 14, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

# Clean the data - bronze tables to silver

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean.limit(6))





StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 845fb7c3-89d0-49c6-9311-4c3158e68767)

In [6]:
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 8, Finished, Available, Finished, False)

In [10]:
%%sql
select * from orders limit 5

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 10, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [11]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))


StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c59df1ae-0677-42af-955f-37648d1422e9)

In [12]:
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 12, Finished, Available, Finished, False)

In [13]:
%%sql 
select * from payments limit 5

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 13, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [30]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", 
                when(col("payment_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("payment_date"), "yyyy/MM/dd"))
                .when(col("payment_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("payment_date"), "dd-MM-yyyy"))
                .when(col("payment_date").rlike("^\d{8}$"), to_date(col("payment_date"), "yyyyMMdd"))
                .otherwise(to_date(col("payment_date"), "yyyy-MM-dd")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))


StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e65a0e36-3a75-49b3-8067-1ff21627e4ca)

In [33]:
payments_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("silver_payments")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 33, Finished, Available, Finished, False)

In [34]:
%%sql
select * from silver_payments limit 5

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 34, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 9 fields>

In [19]:
%%sql
select * from support limit 5

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 19, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [20]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))



StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c62af7d-8daa-454f-93cd-a3efe5a8ae00)

In [21]:
support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 21, Finished, Available, Finished, False)

In [22]:
%%sql 
select * from web limit 5

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 22, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [23]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))



StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e6254b1-1e71-41c3-a195-79b931b7b1a6)

In [24]:
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 24, Finished, Available, Finished, False)

#### Gold tables

In [35]:
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))



StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c8a69ac-03d9-4773-bffc-e1aa9989bd00)

In [36]:
customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 936f810e-033d-45eb-991e-1fa8f24f5824, 36, Finished, Available, Finished, False)